### RQ2 — Hardware-Holdout Test (EC-NAS 4V Benchmark)

Tests whether energy predictions transfer across GPU hardware for the *same* architectures: 91 architectures from EC-NAS's 4V space (|V|≤4), each measured on 4 different GPUs (Quadro RTX 6000, RTX 3060, RTX 3090, Titan Xp). Unlike RQ1 (different architectures, different families), this holds the architecture fixed and varies only the hardware — a cleaner, narrower test of whether `params/depth/flops`-based predictions generalize across GPU generations.

**Step 1 — data access.** This benchmark exists only as TFRecord/protobuf files (confirmed earlier — no raw-JSON shortcut like the base EC-NAS energy data had). Decoded directly rather than skipped:
- The TFRecord *container* format needs no TensorFlow — it's a documented length-prefixed binary format (`uint64 length, uint32 crc, data, uint32 crc`), implemented here in a few lines of pure Python (CRCs read and discarded, not verified — only the length is needed to split records correctly).
- Each record's payload is a JSON array `[module_hash, epochs, raw_adjacency, raw_operations, raw_metrics]` (confirmed from the vendored `nasbench101.py` loader inspected earlier) — plain JSON, no protobuf needed for this part either.
- `raw_metrics` *is* base64-encoded serialized protobuf (`ModelMetricsEnergy`). Rather than installing TensorFlow + Google's full `nasbench` package for this, only the generated Python class (`model_metrics_energy_pb2.py`) was pulled from the repo and vendored into `src/data/nasbench_proto/` — it's self-contained (a serialized descriptor + `google.protobuf`, no TF dependency).
- The four `.tfrecord` files are Git LFS objects — `raw.githubusercontent.com` serves LFS *pointer* files, not the actual data (confirmed: first attempt downloaded 131-byte pointer files). Re-fetched via `media.githubusercontent.com`, which serves the real LFS content directly.

Raw files: `data/raw/ec_nas/hardware_tfrecords/{quadrortx6000,rtx3060,rtx3090,titanxp}.tfrecord`.

In [ ]:
# IMPORTS

import base64
import json
import struct
import sys

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

sys.path.insert(0, "../../")
from src.data.nasbench_proto import model_metrics_energy_pb2
from src.models import random_forest

TFRECORD_DIR = "../../data/raw/ec_nas/hardware_tfrecords/"
GPU_FILES = {
    "quadrortx6000": "Quadro RTX 6000",
    "rtx3060": "RTX 3060",
    "rtx3090": "RTX 3090",
    "titanxp": "Titan Xp",
}

In [ ]:
# MINIMAL TFRECORD READER (no TensorFlow needed) + PARSE ALL FOUR GPU FILES

def read_tfrecords(path):
    """Length-prefixed TFRecord container reader. CRCs are read and discarded,
    not verified -- only the length field is needed to split records correctly."""
    with open(path, "rb") as f:
        while True:
            length_bytes = f.read(8)
            if len(length_bytes) < 8:
                break
            length = struct.unpack("<Q", length_bytes)[0]
            f.read(4)  # length CRC, skipped
            data = f.read(length)
            f.read(4)  # data CRC, skipped
            yield data


rows = []
for fname, gpu_label in GPU_FILES.items():
    hash_counts = {}
    for record in read_tfrecords(TFRECORD_DIR + f"{fname}.tfrecord"):
        module_hash, epochs, raw_adjacency, raw_operations, raw_metrics = json.loads(record.decode("utf-8"))
        metrics = model_metrics_energy_pb2.ModelMetricsEnergy.FromString(base64.b64decode(raw_metrics))

        hash_counts[module_hash] = hash_counts.get(module_hash, 0) + 1
        repeat_idx = hash_counts[module_hash]
        num_vertices = int(len(raw_adjacency) ** 0.5)

        rows.append({
            "run_id": f"{fname}_{module_hash}_repeat{repeat_idx}",
            "architecture_hash": module_hash,
            "gpu_type": gpu_label,
            "epochs": epochs,
            "params": metrics.trainable_parameters,
            "depth": num_vertices,
            "total_energy_kwh": metrics.total_energy,
        })

raw = pd.DataFrame(rows)
print("total records parsed:", len(raw))
print(raw.groupby("gpu_type").size())

In [ ]:
# CHECK: is epochs actually constant? (required before proceeding)

print("epochs unique values overall:", sorted(raw["epochs"].unique()))
print(raw.groupby(["gpu_type", "epochs"]).size())
print()
print("unique architectures per GPU:")
print(raw.groupby("gpu_type")["architecture_hash"].nunique())
missing_3060 = set(raw[raw["gpu_type"] == "Quadro RTX 6000"]["architecture_hash"]) - set(raw[raw["gpu_type"] == "RTX 3060"]["architecture_hash"])
print("architectures present for Quadro RTX 6000 but missing for RTX 3060:", missing_3060)

**Flag: `epochs` is NOT constant — confirmed 4 values `{4, 12, 36, 108}` per architecture per GPU, 91 (or 88 for RTX 3060) records at each.** This isn't a fixed-benchmark constant, and it changes what "proceeding" means here.

**Provenance check, and why only `epochs == 4` is used from here on.** The EC-NAS paper's own appendix (already read directly in an earlier session, §A "Additional Benchmarks and Metrics"): *"For the 4V and 5V spaces the energy efficiency metrics are derived from direct measurements, independent of surrogate modeling. These datasets were compiled by performing exhaustive model training limited to 4 epochs, with the resource costs for the remaining epochs extrapolated through linear scaling."* So within this hardware-specific 4V file, only the `epochs == 4` rows are real per-GPU hardware measurements — the `12`/`36`/`108` rows are linearly extrapolated, not measured on that GPU at all. This is the same "surrogate vs. real" distinction already applied when building `ec_nas_features.csv` (which used only the genuinely-measured 4-epoch subset and excluded `surrogate`/`linscale` data) — applying it here too for consistency, rather than silently mixing real and extrapolated rows into a "hardware transfer" test.

**Second flag: RTX 3060 is missing 3 of the 91 architectures** (confirmed above) — no measurement exists for those 3 hashes on that GPU at all (not an extrapolation issue, just absent). RTX 3060 comparisons use 88 architectures, not 91.

Filtering to `epochs == 4` from here on.

In [ ]:
# BUILD FEATURE TABLE (epochs==4 only) — same schema as ec_nas_features.csv, plus gpu_type

at4 = raw[raw["epochs"] == 4].copy()

features = pd.DataFrame({
    "run_id": at4["run_id"],
    "architecture_hash": at4["architecture_hash"],
    "gpu_type": at4["gpu_type"],
    "params": at4["params"],
    "depth": at4["depth"],
    "flops": 2 * at4["params"],   # same approximation as ec_nas_features.csv
    "epochs": 4,
    "batch_size": 256,             # fixed, same NAS-Bench-101 source as ec_nas_features.csv
    "target": at4["total_energy_kwh"] * 3_600_000,  # kWh -> joules
})

OUT_PATH = "../../data/processed/ec_nas/ec_nas_4v_hardware_features.csv"
features.to_csv(OUT_PATH, index=False)

print(features.shape)
print(features.groupby("gpu_type").size())
print("nulls:", features.isnull().sum().sum())
features.head()

### Small-n reliability check — before running Step 2

91 architectures is small for a train/test split. Checking two things: whether `params`/`depth`/`flops`/`batch_size` actually vary enough to learn from within this set, and — the more important check — whether an 80/20 within-GPU "ceiling" split gives a *stable* number at all, by running it across multiple random seeds rather than trusting a single split.

In [ ]:
# FEATURE VARIABILITY CHECK

CORE_FEATURES = ["params", "depth", "flops", "epochs", "batch_size"]
TARGET = "target"

q6000 = features[features["gpu_type"] == "Quadro RTX 6000"]
print("Quadro RTX 6000 architecture count:", len(q6000))
print(q6000[CORE_FEATURES].describe())

In [ ]:
# HOLDOUT HELPER (reused for both the ceiling splits and the cross-GPU tests)

def holdout(train_df, test_df, label, verbose=True):
    X_train = train_df[CORE_FEATURES]
    y_train_log = np.log1p(train_df[TARGET])
    X_test = test_df[CORE_FEATURES]
    y_test = test_df[TARGET]

    model = random_forest.build_model()
    model.fit(X_train, y_train_log)
    preds_raw = np.expm1(model.predict(X_test))

    mape = mean_absolute_percentage_error(y_test, preds_raw)
    r2 = r2_score(y_test, preds_raw)
    tau, tau_p = kendalltau(y_test, preds_raw)
    importances = pd.Series(model.feature_importances_, index=CORE_FEATURES).sort_values(ascending=False)

    if verbose:
        print(f"=== {label} ===  train n={len(train_df)}, test n={len(test_df)}")
        print(f"MAPE={mape:.4f}  R2={r2:.4f}  Kendall-Tau={tau:.4f}  (p={tau_p:.2e})")
        print(importances.to_dict())
        print()

    return dict(label=label, mape=mape, r2=r2, tau=tau, importances=importances)


# CEILING STABILITY CHECK — same 80/20 split logic, 7 different random seeds
ceiling_results = []
for seed in [0, 1, 2, 3, 4, 42, 100]:
    tr, te = train_test_split(q6000, test_size=0.2, random_state=seed)
    ceiling_results.append(holdout(tr, te, f"ceiling seed={seed}", verbose=False))

ceiling_taus = [r["tau"] for r in ceiling_results]
ceiling_r2s = [r["r2"] for r in ceiling_results]
print("ceiling Kendall-Tau across seeds:", [f"{t:.3f}" for t in ceiling_taus])
print(f"  mean={np.mean(ceiling_taus):.3f}  std={np.std(ceiling_taus):.3f}  range=[{min(ceiling_taus):.3f}, {max(ceiling_taus):.3f}]")
print("ceiling R2 across seeds:", [f"{r:.3f}" for r in ceiling_r2s])
print(f"  range=[{min(ceiling_r2s):.3f}, {max(ceiling_r2s):.3f}]")

**Flag: the ceiling comparison is unreliable at this sample size, confirmed rather than assumed.** With `test size=19`, Kendall-Tau across 7 seeds ranged from **0.597 to 0.865** (mean 0.747, std 0.085) and R² swung from **-0.126 to 0.910** — a single 80/20 split on 91 architectures does not give a stable estimate of anything. `params`+`flops` alone still carry ~98% of feature importance every time (consistent with every prior RF run here), so the *relationship* the model learns is stable; it's the *evaluation* on an 18-19-row test set that isn't. The mean across seeds is used below as the least-bad available ceiling estimate, but any "% of ceiling" number from here on should be read as having a wide, unquantified error bar — not a precise benchmark.

In [ ]:
# STEP 2 — HARDWARE HOLDOUT: train on Quadro RTX 6000 only, test on each other GPU separately

others = {g: features[features["gpu_type"] == g] for g in ["RTX 3060", "RTX 3090", "Titan Xp"]}

hardware_results = [holdout(q6000, test_df, f"Quadro RTX 6000 -> {gpu_name}") for gpu_name, test_df in others.items()]

In [ ]:
# STEP 3 — RELATIVE TO CEILING

ceiling_tau_mean = np.mean(ceiling_taus)
print(f"using mean ceiling Kendall-Tau = {ceiling_tau_mean:.4f} (± {np.std(ceiling_taus):.3f}, see reliability flag above)")
print()
for r in hardware_results:
    pct = 100 * r["tau"] / ceiling_tau_mean
    print(f"{r['label']:32s}  MAPE={r['mape']:.4f}  R2={r['r2']:>8.4f}  Tau={r['tau']:.4f}  -> {pct:.1f}% of ceiling")

**Result** (executed once already; re-run in VS Code to attach outputs):

| direction | MAPE | R² | Kendall-Tau | % of ceiling (mean=0.747) |
|---|---:|---:|---:|---:|
| Quadro RTX 6000 → RTX 3060 | 0.871 | -3.605 | 0.884 | 118.3% |
| Quadro RTX 6000 → RTX 3090 | 0.540 | 0.207 | 0.677 | 90.6% |
| Quadro RTX 6000 → Titan Xp | **0.058** | **0.957** | **0.878** | 117.4% |

**Two directions "exceed 100% of ceiling" — that's the small-n instability showing up again, not evidence of better-than-self transfer.** Given the ceiling itself ranged from 0.597 to 0.865 across seeds, a cross-GPU tau of 0.88 is inside that same noisy range, not meaningfully above the model's own achievable ceiling. The correct read: RTX 3060 and Titan Xp transfer *rank order* about as well as the (unstably-estimated) within-GPU ceiling; RTX 3090 transfers somewhat worse (90.6%). The percentage framing itself is on shakier ground here than it was for RQ1 (37K/2.8K rows there vs. 88-91 here) — reported as requested, with the caveat attached rather than hidden.

**Absolute-scale accuracy (MAPE/R²) tells a different, more interesting story than rank order does.** Titan Xp transfers cleanly on every metric (R²=0.957, MAPE=5.8%) — a genuinely strong result, not just a rank-order one. RTX 3060 shows the now-familiar pattern from RQ1: strong rank preservation (tau=0.884) alongside a deeply negative R² (-3.605, MAPE 87%) — good relative ordering, systematically miscalibrated absolute scale. RTX 3090 is the weakest on every metric simultaneously (tau=0.677, R²=0.207) — this one doesn't fit the "good rank, bad scale" pattern seen elsewhere; it's just a harder transfer target across the board. Given only 91 architectures per GPU, distinguishing "RTX 3090 is a genuinely harder transfer target" from "this particular architecture set happens to be unlucky for RTX 3090" isn't possible without more data — flagged as a limitation, not resolved here.

`params`+`flops` account for ~98% of feature importance in every run (`depth` ~1-2%, `epochs`/`batch_size` exactly 0 — both constant within any single-GPU training set, same mechanism established in RQ1). No auxiliary features were used here, so the `is_mlp_family`-style masking problem from RQ1 doesn't apply — this is a much cleaner test of a narrower question (does a `params/flops`-based energy model transfer across GPUs for fixed architectures), and it gets a genuinely mixed answer: yes for Titan Xp, partially for RTX 3060 (rank only), weakly for RTX 3090.

### Follow-up 1 — is RTX 3060's poor R² a systematic scale bias?

Same diagnostic used for RQ1's epoch-mismatch finding: predicted vs. actual mean/median, and whether the per-row prediction/actual ratio is tight (systematic bias) or scattered (something else).

In [ ]:
# PREDICTED VS ACTUAL SCALE — RTX 3060 (the originally reported seed=42 run)

model = random_forest.build_model()  # default random_state=42, matches the Step 2 run above
model.fit(q6000[CORE_FEATURES], np.log1p(q6000[TARGET]))
preds_3060 = np.expm1(model.predict(others["RTX 3060"][CORE_FEATURES]))
actual_3060 = others["RTX 3060"][TARGET].values

print(f"RTX 3060 actual:    mean={actual_3060.mean():.0f}  median={np.median(actual_3060):.0f}")
print(f"RTX 3060 predicted: mean={preds_3060.mean():.0f}  median={np.median(preds_3060):.0f}")
print(f"ratio (pred/actual), means:   {preds_3060.mean() / actual_3060.mean():.3f}")
print(f"ratio (pred/actual), medians: {np.median(preds_3060) / np.median(actual_3060):.3f}")

ratios = preds_3060 / actual_3060
print(f"\nper-row pred/actual ratio: min={ratios.min():.3f}  max={ratios.max():.3f}  mean={ratios.mean():.3f}  std={ratios.std():.3f}")

print(f"\nQuadro RTX 6000 (training) target mean: {q6000[TARGET].mean():.0f}")
print(f"RTX 3060 (test) actual target mean:      {actual_3060.mean():.0f}")
print(f"Titan Xp (test) actual target mean:       {others['Titan Xp'][TARGET].mean():.0f}")
print(f"RTX 3090 (test) actual target mean:       {others['RTX 3090'][TARGET].mean():.0f}")

**Confirmed: a systematic multiplicative scale bias, not something else.** RTX 3060 predicted mean (77,271 J) vs. actual mean (42,354 J) — a **1.82x** over-prediction on average, and the **per-row ratio is tight** (mean 1.87, std 0.19 across 88 architectures) rather than scattered — every architecture is over-predicted by roughly the same factor, not randomly. That's the signature of a systematic bias, matching the `KendallTau=0.88`-with-`R²=-3.6` pattern: ranking is preserved because the bias multiplies every prediction similarly, but absolute accuracy is destroyed.

**And the bias has an exact, quantifiable source: the model just reproduces Quadro RTX 6000's own energy scale, and RTX 3060 genuinely uses much less energy for the same architectures.** Comparing mean target values directly:

| GPU | mean target (J) | ratio to Quadro RTX 6000 |
|---|---:|---:|
| Quadro RTX 6000 (training) | 79,995 | 1.000 |
| Titan Xp | 81,127 | **1.014** |
| RTX 3090 | 63,512 | 0.794 |
| RTX 3060 | 42,354 | **0.529** |

This lines up almost exactly with the R² results (0.957, 0.207, -3.605 respectively) and with the RTX 3060 bias ratio (1.82-1.89, the near-inverse of 0.529). **Titan Xp transfers well not because of anything sophisticated the model learned — it's because Titan Xp and Quadro RTX 6000 happen to consume almost identical energy for this architecture set (1.4% apart).** RTX 3060 transfers poorly because it's a substantially more energy-efficient GPU for these architectures (uses ~53% of Quadro's energy) and the model, trained only on Quadro's scale with no hardware-efficiency feature to condition on, has no way to know that. This is the same underlying mechanism as RQ1's epoch-budget finding — a model trained on one fixed hardware/duration context bakes that context's scale into the `params/flops`→energy relationship, with no feature available to correct for a different context.

### Follow-up 2 — are the cross-GPU results themselves stable across seeds?

Same 7 seeds as the ceiling check, but note the difference in what "seed" varies: the ceiling check reseeds the 80/20 **train/test split** on 91 rows (small test set, high variance). The cross-GPU tests always train on the full 91 Quadro rows and test on the full 88-91 rows of the other GPU — there's no split to reseed, so "seed" here varies only the Random Forest's own internal stochasticity (bootstrap sampling, feature subsampling at each split), evaluated on a fixed, full-size test set each time.

In [ ]:
# RE-RUN ALL THREE CROSS-GPU TESTS ACROSS THE SAME 7 SEEDS

SEEDS = [0, 1, 2, 3, 4, 42, 100]
seeded_results = {gpu: {"mape": [], "r2": [], "tau": []} for gpu in others}

for seed in SEEDS:
    model = random_forest.build_model(random_state=seed)
    model.fit(q6000[CORE_FEATURES], np.log1p(q6000[TARGET]))
    for gpu_name, test_df in others.items():
        preds_raw = np.expm1(model.predict(test_df[CORE_FEATURES]))
        y_test = test_df[TARGET]
        seeded_results[gpu_name]["mape"].append(mean_absolute_percentage_error(y_test, preds_raw))
        seeded_results[gpu_name]["r2"].append(r2_score(y_test, preds_raw))
        seeded_results[gpu_name]["tau"].append(kendalltau(y_test, preds_raw)[0])

print(f"{'GPU':10s}{'metric':6s}{'min':>10s}{'max':>10s}{'mean':>10s}{'std':>10s}")
for gpu_name, res in seeded_results.items():
    for metric in ["mape", "r2", "tau"]:
        vals = res[metric]
        print(f"{gpu_name:10s}{metric.upper():6s}{min(vals):>10.4f}{max(vals):>10.4f}{np.mean(vals):>10.4f}{np.std(vals):>10.4f}")

**Result across 7 seeds** (executed once already; re-run in VS Code to attach outputs):

| GPU | metric | min | max | mean | std |
|---|---|---:|---:|---:|---:|
| RTX 3060 | MAPE | 0.8707 | 0.8744 | 0.8726 | 0.0012 |
| RTX 3060 | R² | -3.6281 | -3.5891 | -3.6109 | 0.0134 |
| RTX 3060 | Tau | 0.8807 | 0.8886 | 0.8849 | 0.0025 |
| RTX 3090 | MAPE | 0.5383 | 0.5412 | 0.5396 | 0.0009 |
| RTX 3090 | R² | 0.2044 | 0.2071 | 0.2055 | 0.0010 |
| RTX 3090 | Tau | 0.6758 | 0.6787 | 0.6773 | 0.0009 |
| Titan Xp | MAPE | 0.0574 | 0.0580 | 0.0576 | 0.0002 |
| Titan Xp | R² | 0.9568 | 0.9601 | 0.9586 | 0.0010 |
| Titan Xp | Tau | 0.8727 | 0.8776 | 0.8748 | 0.0018 |

**These results turn out to be highly stable — the opposite finding from the ceiling check, for a specific, identifiable reason, not a coincidence.** Every metric's range across 7 seeds is tiny (R² std ≤0.013, MAPE std ≤0.001, Tau std ≤0.0025) compared to the ceiling's swings (Tau std 0.085, R² range spanning more than 1.0). The mechanism difference explains this directly: the ceiling check reseeds an 80/20 **split** of 91 rows, so the *test set itself* changes size-19 composition each time — a small, high-variance sample. The cross-GPU tests always train on the same full 91 rows and evaluate on the same full 88-91 rows; reseeding only changes the Random Forest's internal bootstrap/feature-subsampling choices, which barely moves predictions when averaged over a fixed 88-91-row evaluation set. **So the headline holdout numbers reported earlier were already reliable — it was specifically the ceiling estimate, not the cross-GPU results, that needed the multi-seed treatment.** The original single-run numbers (MAPE 0.871/0.540/0.058, R² -3.605/0.207/0.957, Tau 0.884/0.677/0.878 for RTX 3060/RTX 3090/Titan Xp respectively) all sit well within these seed ranges and can be reported as final without the same caveat that applies to the ceiling.

### Adding real hardware-spec features

**`NVIDIA_GPU_Processors_curated.csv` (the file named for this join) turned out to have two real gaps, checked directly before building anything:** no memory-bandwidth column at all (only pixel/texture fillrate, FP32 GFLOPS, TDP), and RTX 3060/RTX 3090 don't appear anywhere in its 532 rows — it only goes up through the RTX 2000-series plus professional Ampere cards (RTX A2000/A4000/A5000/A6000, different products). Quadro RTX 6000 and Titan Xp are present, but Titan Xp has two conflicting duplicate rows (10790.4 vs. 11366.4 GFLOPS) and Quadro RTX 6000's TDP (180W) doesn't match any commonly-cited figure.

**Sourced externally instead, per direction given when this was flagged:**
- **Quadro RTX 6000**: NVIDIA's own official datasheet (fetched directly) — 16.3 TFLOPS FP32, 672 GB/s memory bandwidth, 260W "total graphics power" (the standard TDP figure; the datasheet separately lists 295W "total board power," a different, higher figure — 260W used here as the conventional TDP citation). This also resolves the curated CSV's incorrect 180W.
- **RTX 3060 / RTX 3090**: Wikipedia's GeForce 30-series specification table — RTX 3060: 12.74 TFLOPS FP32 (boost), 360 GB/s, 170W. RTX 3090: 35.58 TFLOPS FP32 (boost), 936 GB/s, 350W.
- **Titan Xp**: Wikipedia's GeForce 10-series specification table — 12.15 TFLOPS FP32 (boost), 547.7 GB/s, 250W (this TDP matches the curated CSV exactly, and resolves the file's own duplicate-GFLOPS ambiguity in favor of the boost-clock figure).

All four use boost-clock FP32 figures for consistency (matching how "peak FLOPs" is normally reported).

In [ ]:
# GPU SPECS TABLE (externally sourced, see markdown above) + JOIN + arithmetic_intensity_ratio

gpu_specs = pd.DataFrame({
    "gpu_type": ["Quadro RTX 6000", "RTX 3060", "RTX 3090", "Titan Xp"],
    "peak_flops_gflops": [16300, 12740, 35580, 12150],   # FP32, boost clock
    "memory_bandwidth_gbs": [672, 360, 936, 547.7],
    "tdp_watts": [260, 170, 350, 250],
})
print(gpu_specs)

features = features.merge(gpu_specs, on="gpu_type", how="left")
assert features["peak_flops_gflops"].notnull().all(), "unmatched gpu_type found"

# flops / peak_flops: how much of that GPU's peak FP32 capacity this architecture needs
features["arithmetic_intensity_ratio"] = features["flops"] / (features["peak_flops_gflops"] * 1e9)

OUT_PATH = "../../data/processed/ec_nas/ec_nas_4v_hardware_features.csv"
features.to_csv(OUT_PATH, index=False)

print(features.shape)
features.head()

In [ ]:
# RE-RUN THE THREE CROSS-GPU TESTS WITH HARDWARE-SPEC FEATURES ADDED

HW_FEATURES = CORE_FEATURES + ["peak_flops_gflops", "memory_bandwidth_gbs", "arithmetic_intensity_ratio"]


def holdout_hw(train_df, test_df, feats, label):
    X_train = train_df[feats]
    y_train_log = np.log1p(train_df["target"])
    X_test = test_df[feats]
    y_test = test_df["target"]

    model = random_forest.build_model()
    model.fit(X_train, y_train_log)
    preds_raw = np.expm1(model.predict(X_test))

    mape = mean_absolute_percentage_error(y_test, preds_raw)
    r2 = r2_score(y_test, preds_raw)
    tau, tau_p = kendalltau(y_test, preds_raw)
    importances = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)

    print(f"=== {label} ===  train n={len(train_df)}, test n={len(test_df)}")
    print(f"MAPE={mape:.4f}  R2={r2:.4f}  Kendall-Tau={tau:.4f}  (p={tau_p:.2e})")
    print(importances.to_dict())
    print(f"  predicted mean={preds_raw.mean():.0f}  actual mean={y_test.mean():.0f}  ratio={preds_raw.mean()/y_test.mean():.3f}")
    print()

    return dict(label=label, mape=mape, r2=r2, tau=tau, importances=importances,
                pred_mean=preds_raw.mean(), actual_mean=y_test.mean())


q6000 = features[features["gpu_type"] == "Quadro RTX 6000"]
others_hw = {g: features[features["gpu_type"] == g] for g in ["RTX 3060", "RTX 3090", "Titan Xp"]}

hw_results = [holdout_hw(q6000, test_df, HW_FEATURES, f"Quadro RTX 6000 -> {gpu_name} (with HW specs)")
              for gpu_name, test_df in others_hw.items()]

**Result — and the direct answer to "does the Quadro-mimicking bias persist": yes, essentially unchanged, and the reason is visible directly in feature importance.**

| direction | MAPE | R² | Tau | pred mean | actual mean | ratio |
|---|---:|---:|---:|---:|---:|---:|
| → RTX 3060 (core-only) | 0.871 | -3.605 | 0.884 | 77,271 | 42,354 | 1.824 |
| → RTX 3060 (+ HW specs) | **0.962** | **-4.410** | **0.851** | 80,331 | 42,354 | **1.897** |
| → RTX 3090 (core-only) | 0.540 | 0.207 | 0.677 | 79,534 | 63,512 | 1.252 |
| → RTX 3090 (+ HW specs) | 0.459 | 0.213 | 0.594 | 71,425 | 63,512 | 1.125 |
| → Titan Xp (core-only) | 0.058 | 0.957 | 0.878 | 79,534 | 81,127 | 0.980 |
| → Titan Xp (+ HW specs) | 0.097 | **0.898** | **0.792** | 82,515 | 81,127 | 1.017 |

**`peak_flops_gflops` and `memory_bandwidth_gbs` — the two features this task asked to add — got exactly `0.0` importance in every single run.** Only `arithmetic_intensity_ratio` picked up nonzero importance (0.319) — worth being precise about why, since it looks like a partial win but isn't: this is **the identical mechanism as `is_mlp_family` in RQ1**. Training happens on Quadro RTX 6000 *only*, so `peak_flops_gflops` and `memory_bandwidth_gbs` are **constant** across every training row (both always equal Quadro's own values) — a Random Forest cannot split on a zero-variance feature, exactly as established before. `arithmetic_intensity_ratio` only appears to help because `= flops / (constant peak_flops)` is just `flops` rescaled by a constant within this training set — it is not new information, it's `flops` split across two correlated columns. This is confirmed by the results actually getting **worse**, not better, on most metrics: RTX 3060's R² dropped from -3.605 to -4.410 and Tau from 0.884 to 0.851; Titan Xp's R² dropped from 0.957 to 0.898 and Tau from 0.878 to 0.792. Only RTX 3090 improved marginally (R² 0.207→0.213), well within noise.

**Predicted-vs-actual means confirm the bias didn't move toward the target GPU's real scale — if anything it moved slightly further away for the hardest case.** RTX 3060's predicted mean went from 77,271 (core-only) to 80,331 (with HW specs) against an actual mean of 42,354 — the ratio got *worse* (1.824 → 1.897), not better. RTX 3090 moved somewhat closer (ratio 1.252 → 1.125) but is still substantially overshooting. Titan Xp, which was already accurate, stayed accurate (ratio 0.980 → 1.017). **None of this is the model learning to condition on which GPU it's predicting for — it's the same Quadro-scale prediction as before, with essentially no shift attributable to the new hardware-spec features**, because those features carry zero information under strict single-GPU training, structurally, not due to a modeling oversight.

**This is a second, independent confirmation of the exact same limitation `is_mlp_family` revealed for RQ1: no per-row indicator or spec value can fix a strict single-hardware-context holdout, because the model never observes that value varying during training.** Making this transfer actually work would require either training on more than one GPU (so hardware-spec features have real training variance) or a fundamentally different approach (e.g. a physically-derived scaling law relating known hardware ratios to expected energy, applied post-hoc rather than learned) — not attempted here, flagged as the next real question if RQ2 needs to move past this ceiling.

### Habitat-style physics-based scaling baseline

No training, no model — a direct formula, per the "no-training-needed" baseline concept from Habitat (Yu et al. 2021, cross-GPU runtime scaling from a measured baseline + hardware-spec ratio):

`predicted_energy(target_gpu) = quadro_energy × (quadro_peak_flops / target_gpu_peak_flops)`

For each architecture, take its *measured* Quadro RTX 6000 energy and scale it by the ratio of peak FP32 FLOPs between Quadro and the target GPU. This directly tests whether the assumption "energy scales inversely with peak compute throughput" holds for this workload — worth noting up front: since scaling by a positive constant never changes rank order, this formula's Kendall-Tau is mathematically guaranteed to just reflect how well Quadro's own architecture ranking transfers directly, regardless of whether the scaling *magnitude* is right. It only tests the RF's ability to get absolute scale right, not ranking.

In [ ]:
# DIRECT CALCULATION — no training, joins Quadro's own measured energy per architecture
# to each target GPU's actual measurement, scales by the peak-FLOPs ratio

q6000_lookup = features[features["gpu_type"] == "Quadro RTX 6000"][
    ["architecture_hash", "target", "peak_flops_gflops"]
].rename(columns={"target": "quadro_energy", "peak_flops_gflops": "quadro_peak_flops"})

habitat_results = []
for gpu_name in ["RTX 3060", "RTX 3090", "Titan Xp"]:
    target_lookup = features[features["gpu_type"] == gpu_name][
        ["architecture_hash", "target", "peak_flops_gflops"]
    ].rename(columns={"target": "actual_energy", "peak_flops_gflops": "target_peak_flops"})

    merged = q6000_lookup.merge(target_lookup, on="architecture_hash", how="inner")
    merged["predicted_energy"] = merged["quadro_energy"] * (merged["quadro_peak_flops"] / merged["target_peak_flops"])

    mape = mean_absolute_percentage_error(merged["actual_energy"], merged["predicted_energy"])
    r2 = r2_score(merged["actual_energy"], merged["predicted_energy"])
    tau, tau_p = kendalltau(merged["actual_energy"], merged["predicted_energy"])
    pred_mean = merged["predicted_energy"].mean()
    actual_mean = merged["actual_energy"].mean()
    scale_factor = merged["quadro_peak_flops"].iloc[0] / merged["target_peak_flops"].iloc[0]

    print(f"=== Quadro RTX 6000 -> {gpu_name} (Habitat-style scaling) ===  n={len(merged)}")
    print(f"MAPE={mape:.4f}  R2={r2:.4f}  Kendall-Tau={tau:.4f}  (p={tau_p:.2e})")
    print(f"scaling factor (quadro_peak/target_peak) = {scale_factor:.4f}")
    print(f"predicted mean={pred_mean:.0f}  actual mean={actual_mean:.0f}  ratio={pred_mean/actual_mean:.3f}")
    print()

    habitat_results.append(dict(gpu=gpu_name, mape=mape, r2=r2, tau=tau, pred_mean=pred_mean,
                                 actual_mean=actual_mean, scale_factor=scale_factor, n=len(merged)))

**Result — the physics-based baseline is worse than both RF variants on every GPU by R², despite requiring no training at all.**

| direction | method | MAPE | R² | Tau | pred/actual ratio |
|---|---|---:|---:|---:|---:|
| → RTX 3060 | RF (core-only) | 0.871 | -3.605 | 0.884 | 1.824 |
| → RTX 3060 | RF (+HW specs) | 0.962 | -4.410 | 0.851 | 1.897 |
| → RTX 3060 | **Habitat scaling** | 1.395 | **-11.501** | 0.847 | 2.344 |
| → RTX 3090 | RF (core-only) | 0.540 | 0.207 | 0.677 | 1.252 |
| → RTX 3090 | RF (+HW specs) | 0.459 | 0.213 | 0.594 | 1.125 |
| → RTX 3090 | **Habitat scaling** | 0.323 | **0.002** | 0.678 | 0.577 |
| → Titan Xp | RF (core-only) | 0.058 | 0.957 | 0.878 | 0.980 |
| → Titan Xp | RF (+HW specs) | 0.097 | 0.898 | 0.792 | 1.017 |
| → Titan Xp | **Habitat scaling** | 0.318 | **0.024** | 0.885 | 1.323 |

**Kendall-Tau behaves exactly as predicted going in — it's essentially just "does Quadro's own architecture ranking transfer," not a real test of the scaling law.** All three Habitat-scaling Tau values (0.847, 0.678, 0.885) are close to their corresponding RF values, as expected from a constant-multiplier transform preserving rank order by construction.

**R² tells the real story, and it's a clear negative result for the peak-FLOPs-ratio assumption on this workload.** Habitat scaling is dramatically worse than RF for RTX 3060 (R²=-11.5 vs. -3.6/-4.4) and, more strikingly, converts Titan Xp from RF's *best* result (R²=0.957) into Habitat's *worst-in-relative-terms* one (R²=0.024, barely better than predicting the mean). Comparing the formula's scaling factor to the actual empirical energy ratio makes the mismatch concrete:

| GPU | FLOPs-ratio scaling factor used | actual empirical ratio (Quadro mean / target mean) |
|---|---:|---:|
| RTX 3060 | 1.279 | 1.888 |
| RTX 3090 | 0.458 | 1.259 |
| Titan Xp | 1.342 | 0.986 |

**Peak FP32 FLOPs ratio is a poor proxy for actual relative energy consumption across these four GPUs on this specific workload.** RTX 3090 is the clearest case: its peak FLOPs are more than double Quadro's (scaling factor 0.458, i.e. "should" use less than half the energy), but its real energy is only modestly lower (ratio 1.259, i.e. ~79% of Quadro's) — nowhere close to the FLOPs ratio's prediction. The likely explanation: these are small NAS-Bench-101 cell architectures (a handful of conv/pool operations at 4 vertices), which plausibly don't come close to saturating any of these GPUs' peak theoretical throughput — so real-world energy differences are driven more by base/idle power draw, clock behavior, and memory access patterns than by peak compute capacity. This is a workload-specific finding (small cells, not large-scale training), not a claim that Habitat-style scaling is wrong in general.

**Bottom line for RQ2:** neither approach transfers hardware context reliably from a single training GPU. The learned RF at least gets Titan Xp very right (its scale happens to be close to Quadro's); the physics-based formula gets none of the three right, because the specific physical assumption it encodes doesn't hold for this workload. Both point the same direction: single-GPU training data — learned or formula-based — isn't enough to predict absolute energy on a different GPU here; what's missing is either multi-GPU training data or a scaling relationship actually validated against this class of small-cell workload, not proxy compute specs.